# 📦 Step 1: Tokenize Training Data

**Purpose:** Tokenize 10M samples ONCE, then download for reuse

- **Input:** `training_merged.rar` (raw text files)
- **Output:** `tokenized_data.zip` (processed dataset)
- **Time:** 1-2 hours

---

## Benefits

✅ Tokenize once, train multiple times
✅ Experiment with different training configs
✅ No need to re-tokenize for each experiment

---

## Instructions

1. Upload `training_merged.rar`
2. Run all cells
3. Download `tokenized_data.zip` at the end
4. Use in training notebook!

## 1️⃣ Check GPU (Optional - CPU is fine for tokenization)

In [ ]:
!nvidia-smi 2>/dev/null || echo 'No GPU - using CPU (fine for tokenization)'

## 2️⃣ Check RAM & Install Dependencies

In [ ]:
import psutil

ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"💾 Total RAM: {ram_gb:.1f} GB")

if ram_gb < 12:
    print("⚠️  WARNING: Less than 12GB RAM. Tokenization may be slow.")
else:
    print("✅ Sufficient RAM for tokenization")

In [ ]:
!pip install -q transformers datasets sentencepiece psutil

## 3️⃣ Upload & Extract Training Data

Upload `training_merged.rar` using the Files panel (📁)

In [ ]:
import os

# Install unrar
!apt-get install -qq unrar

# Auto-detect compressed file
files = os.listdir('.')
compressed_file = None

for f in files:
    if f.startswith('training_merged'):
        compressed_file = f
        break

if compressed_file:
    print(f"📦 Found: {compressed_file}")
    
    if compressed_file.endswith('.zip'):
        !unzip -q {compressed_file} -d data/
    elif compressed_file.endswith('.rar'):
        !unrar x -y {compressed_file} data/
    elif compressed_file.endswith('.tar.gz') or compressed_file.endswith('.tgz'):
        !tar -xzf {compressed_file} -C data/
    
    print("\n✅ Extraction complete!")
    print("\n📁 Files:")
    !ls -lh data/training_merged/ | head -n 35
else:
    print("❌ No training_merged file found!")
    print("Please upload: training_merged.rar")

## 4️⃣ Load Dataset (Memory-Efficient)

This uses disk caching to avoid RAM issues.

In [ ]:
from pathlib import Path
from datasets import load_dataset

TRAIN_DATA_DIR = Path("data/training_merged")

print("📥 Scanning training data...")
train_files = sorted(TRAIN_DATA_DIR.glob("*.txt"))

print(f"\n📁 Found {len(train_files)} language files:")
total_size = 0
for file_path in train_files:
    file_size_mb = file_path.stat().st_size / (1024 * 1024)
    total_size += file_size_mb
    print(f"   ✅ {file_path.stem:20} - {file_size_mb:6.1f} MB")

print(f"\n📊 Total size: {total_size:.1f} MB")

print("\n🔄 Creating dataset from text files...")
train_dataset = load_dataset(
    'text',
    data_files={'train': [str(f) for f in train_files]},
    split='train',
    cache_dir='./cache'
)

print(f"\n✅ Dataset created with {len(train_dataset):,} examples")
print(f"📊 Total languages: {len(train_files)}")

## 5️⃣ Optional: Sample Data for Faster Training

**Choose your strategy:**
- **100%** - Best quality, longest training (100+ hours)
- **30%** - Good quality, overnight training (~24-30 hours)
- **10%** - Fast testing, ~8-12 hours

In [ ]:
# SET YOUR SAMPLING PERCENTAGE HERE
SAMPLE_PERCENTAGE = 1.0  # 1.0 = 100%, 0.3 = 30%, 0.1 = 10%

if SAMPLE_PERCENTAGE < 1.0:
    original_size = len(train_dataset)
    sample_size = int(original_size * SAMPLE_PERCENTAGE)
    
    print(f"🎲 Sampling {SAMPLE_PERCENTAGE*100}% of data...")
    train_dataset = train_dataset.shuffle(seed=42).select(range(sample_size))
    
    print(f"📊 Reduced from {original_size:,} to {len(train_dataset):,} samples")
    print(f"   (Still includes all 29 languages proportionally)")
else:
    print(f"✅ Using 100% of data ({len(train_dataset):,} samples)")

## 6️⃣ Setup Tokenizer

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "bigscience/bloomz-560m"
MAX_LENGTH = 256

print(f"📥 Loading tokenizer: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer loaded")
print(f"   Max length: {MAX_LENGTH} tokens")

## 7️⃣ Tokenize Dataset

⚠️ **This takes 30-90 minutes depending on data size!**

In [ ]:
import gc
from datetime import datetime

print(f"🔤 Tokenizing {len(train_dataset):,} samples...")
print(f"⏱️  Started: {datetime.now().strftime('%H:%M:%S')}")
print("⚠️  This will take 30-90 minutes. Don't interrupt!\n")

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length'
    )

tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=1000,
    remove_columns=["text"],
    num_proc=2,
    desc="Tokenizing"
)

# Clear memory
del train_dataset
gc.collect()

print(f"\n✅ Tokenization complete!")
print(f"⏱️  Finished: {datetime.now().strftime('%H:%M:%S')}")

## 8️⃣ Split Train/Validation

In [ ]:
import gc

print(f"📊 Splitting train/validation (90/10)...")
split_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_data = split_dataset['train']
val_data = split_dataset['test']

# Clear memory
del tokenized_dataset, split_dataset
gc.collect()

print(f"   ✅ Train: {len(train_data):,} samples")
print(f"   ✅ Val:   {len(val_data):,} samples")

## 9️⃣ Save Tokenized Data to Disk

We'll save in Arrow format (fast & compact)

In [ ]:
import os

OUTPUT_DIR = "tokenized_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"💾 Saving tokenized data to {OUTPUT_DIR}/...")

# Save datasets
train_data.save_to_disk(f"{OUTPUT_DIR}/train")
val_data.save_to_disk(f"{OUTPUT_DIR}/val")

# Save tokenizer
tokenizer.save_pretrained(f"{OUTPUT_DIR}/tokenizer")

print("\n✅ Saved:")
print(f"   📁 {OUTPUT_DIR}/train/ - Training data")
print(f"   📁 {OUTPUT_DIR}/val/ - Validation data")
print(f"   📁 {OUTPUT_DIR}/tokenizer/ - Tokenizer config")

## 🔟 Create Download ZIP

Compress for easy download

In [ ]:
print("📦 Creating ZIP file...")
!zip -r -q tokenized_data.zip tokenized_data/

# Check size
import os
size_mb = os.path.getsize('tokenized_data.zip') / (1024 * 1024)

print(f"\n✅ Created: tokenized_data.zip ({size_mb:.1f} MB)")
print("\n📥 Download instructions:")
print("   1. Click Files panel (📁) on the left")
print("   2. Right-click 'tokenized_data.zip'")
print("   3. Select 'Download'")
print("\n🎉 Use this file in the training notebook!")

---

## ✅ **DONE!**

### Next Steps:

1. **Download** `tokenized_data.zip` (see instructions above)
2. **Save it** on your PC
3. **Upload it** to the training notebook
4. **Train** without re-tokenizing!

---

### Summary:

| Item | Value |
|------|-------|
| Languages | 29 |
| Train samples | (see above) |
| Val samples | (see above) |
| Max length | 256 tokens |
| Ready for | Training! |
